# Day 3-03｜Tracking-to-BEV Mini Project

> Python「黑客松」實戰分析籃球運動數據  
> Notebook 以初學 Python 學員為主：先觀察、再改變少量變數、最後才串接。

目標：把 track_id 的 footpoint 投影到 BEV，並畫出每個 ID 的移動路徑。這是 Day 1～3 的整合收尾。

In [ ]:
from pathlib import Path
import sys

# 在 Colab 中先掛載 Drive；本機執行時會自動略過。
try:
    from google.colab import drive

    drive.mount("/content/drive")
except Exception:
    pass

COURSE_ROOT = Path("/content/drive/MyDrive/basketball_hackathon/course")
if not COURSE_ROOT.exists():
    # 本機或 zip 解壓後執行時，從目前資料夾往上找。
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "assets").exists():
            COURSE_ROOT = candidate
            break

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

print("COURSE_ROOT =", COURSE_ROOT)
print("assets =", COURSE_ROOT / "assets")

In [ ]:
import cv2
import numpy as np
import pandas as pd
from src.cv_utils import (
    load_json,
    read_image_rgb,
    show_image,
    bottom_center,
    save_image_rgb,
)
from src.geometry_utils import compute_homography, project_points

tracks_path = COURSE_ROOT / "assets" / "results" / "d3_02_tracks.json"
if tracks_path.exists():
    records = load_json(tracks_path)
else:
    print(
        "找不到 d3_02_tracks.json，先使用 sample_tracking_boxes 重新產生簡單 track。請優先跑 d3_02。"
    )
    records = []

if not records:
    track_data = load_json(
        COURSE_ROOT / "assets" / "samples" / "sample_tracking_boxes.json"
    )
    for f in track_data["frames"]:
        for i, d in enumerate(f["detections"]):
            records.append(
                {"frame": f["frame_index"], "track_id": i, "bbox_xyxy": d["bbox_xyxy"]}
            )

homo = load_json(COURSE_ROOT / "assets" / "samples" / "sample_homography_points.json")
H = compute_homography(homo["camera_points"], homo["bev_points"])

rows = []
for r in records:
    foot = bottom_center(r["bbox_xyxy"])
    bev_pt = project_points([foot], H)[0]
    rows.append(
        {
            **r,
            "foot_x": foot[0],
            "foot_y": foot[1],
            "bev_x": float(bev_pt[0]),
            "bev_y": float(bev_pt[1]),
        }
    )
tracks = pd.DataFrame(rows)
tracks.head()

In [ ]:
bev = read_image_rgb(COURSE_ROOT / "assets" / "samples" / "sample_bev_court.png")
canvas = bev.copy()

for tid, g in tracks.groupby("track_id"):
    pts = g.sort_values("frame")[["bev_x", "bev_y"]].to_numpy()
    pts_int = np.round(pts).astype(int)
    for a, b in zip(pts_int[:-1], pts_int[1:]):
        cv2.line(canvas, tuple(a), tuple(b), (80, 80, 80), 2)
    if len(pts_int):
        cv2.circle(canvas, tuple(pts_int[-1]), 7, (255, 80, 80), -1)
        cv2.putText(
            canvas,
            str(tid),
            tuple(pts_int[-1] + np.array([8, -8])),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (20, 20, 20),
            2,
        )

show_image(canvas, "BEV track paths")

save_path = COURSE_ROOT / "assets" / "results" / "d3_03_tracking_to_bev_paths.png"
save_image_rgb(save_path, canvas)
print("saved:", save_path)

Day 1～3 到這裡完成：座標 → Homography → Detection → Tracking → BEV。Day 4 開始換成近距離投籃影片。